### Install + setup


In [ ]:
!pip install -q torch torchvision datasets transformers numpy pandas matplotlib scikit-learn tqdm scipy

import os
import json
import glob
import math
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from IPython.display import display
from google.colab import files

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

dirs = [
    "transformer_project/data",
    "transformer_project/models",
    "transformer_project/utils",
    "transformer_project/experiments",
    "transformer_project/results"
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

BASE_DIR = Path("/content/prof_agnews_outputs")
ANALYSIS_DIR = Path("/content/agnews_final_analysis")

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir(parents=True, exist_ok=True)

if ANALYSIS_DIR.exists():
    shutil.rmtree(ANALYSIS_DIR)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device : {DEVICE}")
print(f"Seed   : {SEED}")
print(f"PyTorch: {torch.__version__}")
print("BASE_DIR     :", BASE_DIR)
print("ANALYSIS_DIR :", ANALYSIS_DIR)

### Generic scratch data loader for AG News + DBPedia

In [ ]:
%%writefile transformer_project/data/data_loader_scratch.py
"""
Generic scratch-model data loader for AG News and DBPedia.
Supports:
- official train/test split
- validation split from training only
- train-size fractions
- simple evaluation-time shifts
"""

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import torch
import numpy as np


def get_dataset_spec(dataset_name: str):
    if dataset_name == "ag_news":
        return {
            "hf_name": "ag_news",
            "text_field": "text",
            "num_classes": 4
        }
    elif dataset_name == "dbpedia_14":
        return {
            "hf_name": "dbpedia_14",
            "text_field": "content",
            "num_classes": 14
        }
    else:
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")


def build_vocab(texts, min_freq=2, max_vocab_size=30000):
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.most_common(max_vocab_size - 2):
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab


def apply_shift_to_text(text, shift_config=None, idx=0):
    if shift_config is None:
        return text

    shift_name = shift_config.get("name", "clean")
    if shift_name == "clean":
        return text

    tokens = text.lower().split()

    if shift_name == "truncation":
        max_tokens = shift_config.get("max_tokens", 32)
        tokens = tokens[:max_tokens]
        return " ".join(tokens)

    if shift_name == "unk_corruption":
        rate = shift_config.get("rate", 0.2)
        seed = shift_config.get("seed", 42)
        rng = np.random.RandomState(seed + idx)

        if len(tokens) == 0:
            return text

        n_corrupt = int(round(rate * len(tokens)))
        n_corrupt = min(n_corrupt, len(tokens))

        if n_corrupt > 0:
            corrupt_idx = rng.choice(len(tokens), size=n_corrupt, replace=False)
            for j in corrupt_idx:
                tokens[j] = "__UNKSHIFT__"

        return " ".join(tokens)

    raise ValueError(f"Unknown shift name: {shift_name}")


def text_to_indices(text, vocab, max_length=128):
    tokens = text.lower().split()[:max_length]
    indices = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    padding_length = max_length - len(indices)
    if padding_length > 0:
        indices.extend([vocab["<PAD>"]] * padding_length)

    return indices


class ScratchTextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length=128, shift_config=None):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_length = max_length
        self.shift_config = shift_config

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        shifted_text = apply_shift_to_text(text, self.shift_config, idx=idx)
        input_ids = text_to_indices(shifted_text, self.vocab, self.max_length)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long)
        }


def load_text_dataset(
    dataset_name: str,
    max_length=128,
    split_seed=42,
    val_ratio=0.15,
    train_fraction=1.0
):
    spec = get_dataset_spec(dataset_name)

    print(f"Loading {dataset_name}...")
    dataset = load_dataset(spec["hf_name"])

    official_train = dataset["train"]
    official_test = dataset["test"]

    train_texts_full = [item[spec["text_field"]] for item in official_train]
    train_labels_full = [item["label"] for item in official_train]

    test_texts = [item[spec["text_field"]] for item in official_test]
    test_labels = [item["label"] for item in official_test]

    print(f"Official split sizes: Train={len(train_texts_full):,} | Test={len(test_texts):,}")

    rng = np.random.RandomState(split_seed)
    indices = rng.permutation(len(train_texts_full))

    n_val = int(val_ratio * len(train_texts_full))
    val_indices = indices[:n_val]
    train_indices = indices[n_val:]

    if train_fraction < 1.0:
        frac_rng = np.random.RandomState(split_seed + 1)
        n_train_keep = max(1, int(round(train_fraction * len(train_indices))))
        sampled = frac_rng.choice(train_indices, size=n_train_keep, replace=False)
        train_indices = np.array(sampled)

    train_texts = [train_texts_full[i] for i in train_indices]
    train_labels = [train_labels_full[i] for i in train_indices]

    val_texts = [train_texts_full[i] for i in val_indices]
    val_labels = [train_labels_full[i] for i in val_indices]

    print(
        f"Split (seed={split_seed}, train_fraction={train_fraction:.2f}): "
        f"Train={len(train_texts):,} | Val={len(val_texts):,} | Test={len(test_texts):,}"
    )

    vocab = build_vocab(train_texts, min_freq=2, max_vocab_size=30000)
    print(f"Vocabulary size: {len(vocab):,} (built from final training subset only)")

    train_dataset = ScratchTextDataset(train_texts, train_labels, vocab, max_length=max_length, shift_config=None)
    val_dataset = ScratchTextDataset(val_texts, val_labels, vocab, max_length=max_length, shift_config=None)
    test_dataset = ScratchTextDataset(test_texts, test_labels, vocab, max_length=max_length, shift_config=None)

    return train_dataset, val_dataset, test_dataset, vocab, spec["num_classes"]


def get_dataloaders(train_dataset, val_dataset, test_dataset, batch_size=32):
    pin = torch.cuda.is_available()

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=pin
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=pin
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=pin
    )
    return train_loader, val_loader, test_loader


def build_shifted_test_loader(base_test_dataset, shift_config, batch_size=32):
    shifted_dataset = ScratchTextDataset(
        texts=base_test_dataset.texts,
        labels=base_test_dataset.labels,
        vocab=base_test_dataset.vocab,
        max_length=base_test_dataset.max_length,
        shift_config=shift_config
    )

    pin = torch.cuda.is_available()
    return DataLoader(
        shifted_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=pin
    )

### Transformer model

In [ ]:
%%writefile transformer_project/models/transformer.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, -1e9)

        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 8, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = x + self.drop1(self.self_attn(self.norm1(x), mask))
        x = x + self.drop2(self.feed_forward(self.norm2(x)))
        return x


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int = 128,
                 n_layers: int = 6, n_heads: int = 8, d_ff: int = 2048,
                 max_seq_len: int = 128, dropout: float = 0.1, pad_idx: int = 0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding.weight, mean=0, std=self.d_model**-0.5)
        if self.embedding.padding_idx is not None:
            self.embedding.weight.data[self.embedding.padding_idx].zero_()

        depth_scale = (2 * self.n_layers) ** -0.5

        for layer in self.encoder_layers:
            for module in [layer.self_attn.W_q, layer.self_attn.W_k, layer.self_attn.W_v]:
                nn.init.xavier_uniform_(module.weight)
            nn.init.xavier_uniform_(layer.self_attn.W_o.weight, gain=depth_scale)
            nn.init.zeros_(layer.self_attn.W_o.bias)

            nn.init.xavier_uniform_(layer.feed_forward.linear1.weight)
            nn.init.zeros_(layer.feed_forward.linear1.bias)
            nn.init.xavier_uniform_(layer.feed_forward.linear2.weight, gain=depth_scale)
            nn.init.zeros_(layer.feed_forward.linear2.bias)

        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def create_padding_mask(self, input_ids: torch.Tensor) -> torch.Tensor:
        return (input_ids != self.pad_idx).float()

    def encode(self, input_ids: torch.Tensor) -> torch.Tensor:
        mask = self.create_padding_mask(input_ids)
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.encoder_layers:
            x = layer(x, mask)

        mask_exp = mask.unsqueeze(-1).expand(x.size())
        sum_mask = mask_exp.sum(dim=1).clamp(min=1e-9)
        pooled = torch.sum(x * mask_exp, dim=1) / sum_mask
        return pooled

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        pooled = self.encode(input_ids)
        return self.classifier(self.dropout(pooled))

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def create_model_configs():
    return [
        {"name": "tiny-1",   "d_model": 32,  "n_layers": 1, "n_heads": 2, "d_ff": 128},
        {"name": "small-1",  "d_model": 64,  "n_layers": 1, "n_heads": 4, "d_ff": 256},
        {"name": "small-4",  "d_model": 64,  "n_layers": 4, "n_heads": 4, "d_ff": 256},
        {"name": "medium-4", "d_model": 128, "n_layers": 4, "n_heads": 8, "d_ff": 512},
        {"name": "large-6",  "d_model": 256, "n_layers": 6, "n_heads": 8, "d_ff": 1024},
    ]

### Metrics + CI

In [ ]:
%%writefile transformer_project/utils/metrics_scratch.py
import copy
import torch
import torch.nn as nn
import numpy as np
from scipy.optimize import minimize_scalar


def compute_spectral_norm_power_iteration(weight: torch.Tensor, num_iters: int = 20) -> float:
    if weight.dim() != 2:
        weight = weight.reshape(weight.size(0), -1)

    with torch.no_grad():
        u = torch.randn(weight.size(0), device=weight.device)
        u = u / (u.norm() + 1e-8)

        for _ in range(num_iters):
            v = weight.t() @ u
            v = v / (v.norm() + 1e-8)
            u = weight @ v
            u = u / (u.norm() + 1e-8)

        sigma = (u @ (weight @ v)).item()

    return abs(sigma)


def compute_model_norms(model: nn.Module, exclude_embedding: bool = True):
    norms = []
    for name, module in model.named_modules():
        if exclude_embedding and "embedding" in name.lower():
            continue
        if isinstance(module, nn.Linear):
            weight = module.weight.data.detach().cpu()
            norm = compute_spectral_norm_power_iteration(weight, num_iters=20)
            norms.append(norm)

    product = np.prod(norms) if norms else 1.0
    sum_norms = np.sum(norms) if norms else 0.0

    return {
        "norms": norms,
        "product": float(product),
        "sum": float(sum_norms),
        "count": len(norms)
    }


def compute_rademacher_bound(norms, n_samples: int, d_model: int) -> float:
    product = norms["product"]
    bound = (2.0 / np.sqrt(n_samples)) * product * np.sqrt(d_model)
    return float(bound)


def _classification_error(model: nn.Module, data_loader, device: str) -> float:
    model.eval()
    total = 0
    wrong = 0

    with torch.no_grad():
        for batch in data_loader:
            inputs = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(inputs)
            preds = outputs.argmax(dim=1)

            wrong += (preds != labels).sum().item()
            total += labels.size(0)

    return wrong / total


def compute_pacbayes_bound(model: nn.Module, data_loader, device: str = "cpu",
                           sigma: float = 0.01, delta: float = 0.05,
                           posterior_samples: int = 3):
    model = model.to(device)
    model.eval()

    params = [p for p in model.parameters() if p.requires_grad]
    original_state = copy.deepcopy(model.state_dict())

    squared_norm = 0.0
    num_params = 0
    for p in params:
        squared_norm += (p.detach() ** 2).sum().item()
        num_params += p.numel()

    kl = squared_norm / (2.0 * sigma ** 2)

    risks = []
    for _ in range(posterior_samples):
        sampled_state = {}
        for name, tensor in original_state.items():
            if tensor.dtype.is_floating_point:
                noise = torch.randn_like(tensor) * sigma
                sampled_state[name] = tensor + noise
            else:
                sampled_state[name] = tensor.clone()

        model.load_state_dict(sampled_state)
        risks.append(_classification_error(model, data_loader, device))

    model.load_state_dict(original_state)

    empirical_gibbs_risk = float(np.mean(risks))
    n_samples = len(data_loader.dataset)
    complexity = (kl + np.log((2.0 * np.sqrt(n_samples)) / delta)) / (2.0 * max(1, n_samples - 1))
    bound = empirical_gibbs_risk + np.sqrt(complexity)

    return {
        "bound": float(bound),
        "kl": float(kl),
        "empirical_gibbs_risk": float(empirical_gibbs_risk),
        "sigma": float(sigma),
        "posterior_samples": int(posterior_samples),
        "num_params": int(num_params)
    }


def compute_margin_bound(model: nn.Module, data_loader, device: str = "cpu", gamma: float = 0.1):
    model = model.to(device)
    model.eval()

    classifier_weight = model.classifier.weight.detach()
    weight_norm = torch.norm(classifier_weight, p="fro").item()

    total = 0
    margin_violations = 0
    max_feature_norm = 0.0

    with torch.no_grad():
        for batch in data_loader:
            inputs = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            features = model.encode(inputs)
            logits = model.classifier(features)

            correct_scores = logits[torch.arange(len(labels), device=device), labels]
            competitor_scores = logits.clone()
            competitor_scores[torch.arange(len(labels), device=device), labels] = -float("inf")
            max_wrong_scores = competitor_scores.max(dim=1)[0]

            margins = correct_scores - max_wrong_scores
            margin_violations += (margins <= gamma).sum().item()
            total += labels.size(0)

            batch_feature_norm = torch.norm(features, p=2, dim=1).max().item()
            max_feature_norm = max(max_feature_norm, batch_feature_norm)

    margin_error = margin_violations / total
    complexity_term = (2.0 * weight_norm * max_feature_norm) / (gamma * np.sqrt(total))
    bound = margin_error + complexity_term

    return {
        "bound": float(bound),
        "margin_error": float(margin_error),
        "weight_norm": float(weight_norm),
        "max_feature_norm": float(max_feature_norm),
        "gamma": float(gamma)
    }


def expected_calibration_error(labels: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> float:
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels)

    ece = 0.0
    bin_edges = np.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        bin_mask = (confidences >= bin_edges[i]) & (confidences < bin_edges[i + 1])
        if i == n_bins - 1:
            bin_mask = (confidences >= bin_edges[i]) & (confidences <= bin_edges[i + 1])

        if bin_mask.sum() > 0:
            bin_acc = accuracies[bin_mask].mean()
            bin_conf = confidences[bin_mask].mean()
            bin_weight = bin_mask.sum() / len(labels)
            ece += bin_weight * abs(bin_acc - bin_conf)

    return float(ece)


def multiclass_nll(labels: np.ndarray, probs: np.ndarray) -> float:
    probs = np.clip(probs, 1e-12, 1.0)
    return float(-np.mean(np.log(probs[np.arange(len(labels)), labels])))


def multiclass_brier(labels: np.ndarray, probs: np.ndarray, num_classes: int) -> float:
    one_hot = np.zeros((len(labels), num_classes), dtype=np.float64)
    one_hot[np.arange(len(labels)), labels] = 1.0
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def temperature_scaling(labels: np.ndarray, logits: np.ndarray):
    def nll_loss(T):
        scaled_logits = logits / max(T, 1e-8)
        exp_logits = np.exp(scaled_logits - np.max(scaled_logits, axis=1, keepdims=True))
        probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        probs = np.clip(probs, 1e-12, 1.0)
        log_probs = np.log(probs[np.arange(len(labels)), labels])
        return -np.mean(log_probs)

    result = minimize_scalar(nll_loss, bounds=(0.1, 10.0), method="bounded")
    return float(result.x)


def mean_ci95(values):
    values = np.array(values, dtype=np.float64)
    mean = float(values.mean())
    std = float(values.std(ddof=0))
    n = len(values)
    half_width = 1.96 * std / np.sqrt(max(n, 1))
    return {
        "mean": mean,
        "std": std,
        "ci95_low": float(mean - half_width),
        "ci95_high": float(mean + half_width)
    }

### Unified scratch trainer

In [ ]:
%%writefile transformer_project/utils/trainer_scratch.py
import os
import copy
import math
import torch
import torch.nn as nn


class ScratchTrainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        device="cuda",
        learning_rate=1e-4,
        scheduler_mode="cosine",
        warmup_steps=8000,
        total_steps=50000,
        legacy_initial_lr=1.0,
        checkpoint_dir=None
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.learning_rate = learning_rate
        self.scheduler_mode = scheduler_mode
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.legacy_initial_lr = legacy_initial_lr
        self.current_step = 0
        self.checkpoint_dir = checkpoint_dir
        self.scheduler_trace = []

        if self.checkpoint_dir is not None:
            os.makedirs(self.checkpoint_dir, exist_ok=True)

        init_lr = legacy_initial_lr if scheduler_mode in ["legacy_buggy_warmup", "legacy_warmup_fixed_order"] else learning_rate

        self.optimizer = torch.optim.Adam(
            model.parameters(),
            lr=init_lr,
            betas=(0.9, 0.98),
            eps=1e-9
        )
        self.criterion = nn.CrossEntropyLoss()

        self.best_val_loss = float("inf")
        self.best_model_state = None
        self.patience_counter = 0

    def _checkpoint_path(self, name):
        return os.path.join(self.checkpoint_dir, name)

    def save_checkpoint(self, epoch, is_best=False):
        if self.checkpoint_dir is None:
            return

        state = {
            "epoch": epoch,
            "current_step": self.current_step,
            "model_state": self.model.state_dict(),
            "optimizer_state": self.optimizer.state_dict(),
            "best_val_loss": self.best_val_loss,
            "best_model_state": self.best_model_state,
            "patience_counter": self.patience_counter,
            "scheduler_trace": self.scheduler_trace,
        }

        torch.save(state, self._checkpoint_path("latest.pt"))
        if is_best:
            torch.save(state, self._checkpoint_path("best.pt"))

    def load_latest_checkpoint(self):
        if self.checkpoint_dir is None:
            return 0

        latest_path = self._checkpoint_path("latest.pt")
        if not os.path.exists(latest_path):
            return 0

        ckpt = torch.load(latest_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state"])
        self.optimizer.load_state_dict(ckpt["optimizer_state"])
        self.best_val_loss = ckpt["best_val_loss"]
        self.best_model_state = ckpt["best_model_state"]
        self.patience_counter = ckpt["patience_counter"]
        self.current_step = ckpt["current_step"]
        self.scheduler_trace = ckpt.get("scheduler_trace", [])

        start_epoch = ckpt["epoch"] + 1
        print(f"Resuming from checkpoint: epoch {start_epoch}")
        return start_epoch

    def load_best_checkpoint(self):
        if self.checkpoint_dir is None:
            return False

        best_path = self._checkpoint_path("best.pt")
        if not os.path.exists(best_path):
            return False

        ckpt = torch.load(best_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state"])
        self.best_val_loss = ckpt["best_val_loss"]
        self.best_model_state = ckpt["best_model_state"]
        return True

    def _get_cosine_lr(self):
        if self.current_step < self.warmup_steps:
            scale = self.current_step / max(1, self.warmup_steps)
        else:
            progress = (self.current_step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            scale = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.learning_rate * scale

    def _get_legacy_warmup_lr(self):
        step = max(1, self.current_step)
        d_model = self.model.d_model
        return (d_model ** -0.5) * min(step ** -0.5, step * (self.warmup_steps ** -1.5))

    def _lr_for_current_mode(self):
        if self.scheduler_mode == "constant":
            return self.learning_rate
        if self.scheduler_mode == "cosine":
            return self._get_cosine_lr()
        if self.scheduler_mode in ["legacy_buggy_warmup", "legacy_warmup_fixed_order"]:
            return self._get_legacy_warmup_lr()
        raise ValueError(f"Unknown scheduler_mode: {self.scheduler_mode}")

    def _set_lr(self, lr_value):
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr_value

    def train_epoch(self):
        self.model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for batch in self.train_loader:
            input_ids = batch["input_ids"].to(self.device)
            labels = batch["labels"].to(self.device)

            self.optimizer.zero_grad()
            outputs = self.model(input_ids)
            loss = self.criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            lr_before = float(self.optimizer.param_groups[0]["lr"])

            if self.scheduler_mode == "legacy_buggy_warmup":
                lr_used = lr_before
                self.optimizer.step()
                self.current_step += 1
                lr_after = float(self._lr_for_current_mode())
                self._set_lr(lr_after)
            else:
                self.current_step += 1
                lr_used = float(self._lr_for_current_mode())
                self._set_lr(lr_used)
                self.optimizer.step()
                lr_after = float(self.optimizer.param_groups[0]["lr"])

            self.scheduler_trace.append({
                "step": int(self.current_step),
                "lr_before_step": float(lr_before),
                "lr_used": float(lr_used),
                "lr_after_step": float(lr_after),
                "scheduler_mode": self.scheduler_mode,
            })

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        return total_loss / len(self.train_loader), correct / total

    def evaluate(self, loader, return_predictions=False):
        self.model.eval()
        total_loss = 0.0
        correct = 0
        total = 0
        all_preds = []
        all_probs = []
        all_logits = []
        all_labels = []

        with torch.no_grad():
            for batch in loader:
                input_ids = batch["input_ids"].to(self.device)
                labels = batch["labels"].to(self.device)

                outputs = self.model(input_ids)
                loss = self.criterion(outputs, labels)

                total_loss += loss.item()
                probs = torch.softmax(outputs, dim=1)
                preds = outputs.argmax(dim=1)

                correct += (preds == labels).sum().item()
                total += labels.size(0)

                if return_predictions:
                    all_preds.extend(preds.cpu().numpy())
                    all_probs.extend(probs.cpu().numpy())
                    all_logits.extend(outputs.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

        avg_loss = total_loss / len(loader)
        accuracy = correct / total

        if return_predictions:
            return avg_loss, accuracy, all_preds, all_probs, all_logits, all_labels
        return avg_loss, accuracy

    def train(self, num_epochs=30, early_stopping_patience=5, resume=True):
        start_epoch = self.load_latest_checkpoint() if resume else 0

        for epoch in range(start_epoch, num_epochs):
            print(f"\nEpoch {epoch + 1}/{num_epochs}")

            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc = self.evaluate(self.val_loader)

            current_lr = self.optimizer.param_groups[0]["lr"]

            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | LR: {current_lr:.2e}")
            print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

            is_best = False
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.best_model_state = copy.deepcopy(self.model.state_dict())
                self.patience_counter = 0
                is_best = True
                print("New best model saved")
            else:
                self.patience_counter += 1
                print(f"  Patience: {self.patience_counter}/{early_stopping_patience}")

            self.save_checkpoint(epoch, is_best=is_best)

            if self.patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

        restored = self.load_best_checkpoint()
        if restored:
            print(f"\n{'='*60}")
            print(f"Restoring best model (val_loss={self.best_val_loss:.4f})")
            print('='*60)
        elif self.best_model_state is not None:
            print(f"\n{'='*60}")
            print(f"Restoring best model (val_loss={self.best_val_loss:.4f})")
            print('='*60)
            self.model.load_state_dict(self.best_model_state)

### Upload DBPedia professor result zip

In [ ]:
uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded.keys():
    print("-", name)

### Reset DBPedia folders and unzip professor outputs

In [ ]:
BASE_DIR = Path("/content/prof_dbpedia_outputs")
ANALYSIS_DIR = Path("/content/dbpedia_final_analysis")

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir(parents=True, exist_ok=True)

if ANALYSIS_DIR.exists():
    shutil.rmtree(ANALYSIS_DIR)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

zip_files = list(Path("/content").glob("*.zip"))

db_zip_candidates = []
for z in zip_files:
    name = z.name.lower()
    if "db" in name or "dbpedia" in name or "dp" in name:
        db_zip_candidates.append(z)

if not db_zip_candidates:
    raise FileNotFoundError("No DBPedia zip detected. Please upload the DBPedia result zip.")

print("Detected DBPedia zip candidates:")
for z in db_zip_candidates:
    print("-", z)

db_zip = db_zip_candidates[0]
print("\nUsing DBPedia zip:", db_zip)

with zipfile.ZipFile(db_zip, "r") as zf:
    zf.extractall(BASE_DIR)

print("\nExtracted under:", BASE_DIR)

print("\nDetected result-like folders:")
for p in sorted(BASE_DIR.rglob("*")):
    if p.is_dir() and (
        (p / "run_manifest.json").exists()
        or (p / "all_runs.json").exists()
        or (p / "initial_lr_sweep_summary.json").exists()
    ):
        print("RESULT ROOT:", p)

### Find DBPedia result roots

In [ ]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def find_roots_with_file(base_dir, filename):
    matches = []
    for p in base_dir.rglob("*"):
        if p.is_dir() and (p / filename).exists():
            matches.append(p)
    return sorted(matches)


def read_manifest_if_exists(root):
    path = root / "run_manifest.json"
    if path.exists():
        try:
            return load_json(path)
        except Exception:
            return {}
    return {}


all_manifest_roots = find_roots_with_file(BASE_DIR, "run_manifest.json")
all_runs_roots = find_roots_with_file(BASE_DIR, "all_runs.json")
initial_lr_roots = find_roots_with_file(BASE_DIR, "initial_lr_sweep_summary.json")

full_root = None
ablation_root = None
earlydiag_root = None
initial_lr_root = None

for r in all_manifest_roots:
    manifest = read_manifest_if_exists(r)
    run_tag = str(manifest.get("run_tag", "")).lower()
    dataset_name = str(manifest.get("dataset_name", "")).lower()

    if dataset_name == "dbpedia_14" and "full_dbpedia" in run_tag:
        full_root = r

    if dataset_name == "dbpedia_14" and "ablation_shift" in run_tag:
        ablation_root = r

    if dataset_name == "dbpedia_14" and "earlydiag" in run_tag:
        earlydiag_root = r

    if dataset_name == "dbpedia_14" and "initial_lr" in run_tag:
        initial_lr_root = r

if full_root is None:
    for r in all_manifest_roots:
        if (r / "stable_sweep" / "stable_sweep_summary.json").exists() and (r / "data_size_sweep" / "data_size_sweep_summary.json").exists():
            full_root = r
            break

if ablation_root is None:
    for r in all_manifest_roots:
        if (r / "ablation" / "ablation_summary.json").exists():
            manifest = read_manifest_if_exists(r)
            if manifest.get("run_baselines") is False:
                ablation_root = r
                break

if earlydiag_root is None:
    for r in all_runs_roots:
        if "db" in str(r).lower() or "pedia" in str(r).lower():
            earlydiag_root = r
            break

if initial_lr_root is None:
    for r in initial_lr_roots:
        if "db" in str(r).lower() or "pedia" in str(r).lower():
            initial_lr_root = r
            break

print("DETECTED DBPEDIA ROOTS")
print("=" * 100)
print("FULL ROOT       :", full_root)
print("ABLATION ROOT   :", ablation_root)
print("EARLYDIAG ROOT  :", earlydiag_root)
print("INITIAL-LR ROOT :", initial_lr_root)

if full_root is None:
    raise FileNotFoundError("Could not find DBPedia full scratch result root.")
if ablation_root is None:
    raise FileNotFoundError("Could not find DBPedia ablation+shift result root.")
if earlydiag_root is None:
    raise FileNotFoundError("Could not find DBPedia early diagnostics result root.")
if initial_lr_root is None:
    raise FileNotFoundError("Could not find DBPedia initial-LR sweep result root.")

### Verify DBPedia required files

In [ ]:
required_checks = {
    "full_run_manifest": full_root / "run_manifest.json",
    "full_per_seed": full_root / "per_seed",
    "full_scheduler_traces": full_root / "scheduler_traces",
    "full_stable_summary": full_root / "stable_sweep" / "stable_sweep_summary.json",
    "full_ablation_summary": full_root / "ablation" / "ablation_summary.json",
    "full_data_size_summary": full_root / "data_size_sweep" / "data_size_sweep_summary.json",

    "ablation_run_manifest": ablation_root / "run_manifest.json",
    "ablation_per_seed": ablation_root / "per_seed",
    "ablation_scheduler_traces": ablation_root / "scheduler_traces",
    "ablation_summary": ablation_root / "ablation" / "ablation_summary.json",

    "earlydiag_run_manifest": earlydiag_root / "run_manifest.json",
    "earlydiag_all_runs": earlydiag_root / "all_runs.json",

    "initial_lr_run_manifest": initial_lr_root / "run_manifest.json",
    "initial_lr_per_seed": initial_lr_root / "per_seed",
    "initial_lr_scheduler_traces": initial_lr_root / "scheduler_traces",
    "initial_lr_summary": initial_lr_root / "initial_lr_sweep_summary.json",
}

rows = []
for name, path in required_checks.items():
    rows.append({
        "item": name,
        "path": str(path),
        "exists": path.exists()
    })

check_df = pd.DataFrame(rows)
display(check_df)

check_df.to_csv(ANALYSIS_DIR / "dbpedia_required_file_check.csv", index=False)

missing = check_df[check_df["exists"] == False]
if len(missing) > 0:
    print("\nWARNING: Some expected files/folders are missing:")
    display(missing)
else:
    print("\nAll required DBPedia files/folders found.")

### Analysis helper functions

In [ ]:
def mean_ci95_from_values(values):
    arr = np.array(values, dtype=np.float64)
    arr = arr[~np.isnan(arr)]

    if len(arr) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "n": 0,
        }

    mean = float(arr.mean())
    std = float(arr.std(ddof=0))
    half = 1.96 * std / np.sqrt(max(len(arr), 1))

    return {
        "mean": mean,
        "std": std,
        "ci95_low": float(mean - half),
        "ci95_high": float(mean + half),
        "n": int(len(arr)),
    }


def paired_effect_summary(a_vals, b_vals):
    a = np.array(a_vals, dtype=np.float64)
    b = np.array(b_vals, dtype=np.float64)

    diff = b - a
    mean_diff = float(diff.mean())

    std_diff_sample = float(diff.std(ddof=1)) if len(diff) >= 2 else np.nan
    dz = float(mean_diff / std_diff_sample) if len(diff) >= 2 and std_diff_sample > 0 else np.nan

    std_diff_population = float(diff.std(ddof=0))
    half = 1.96 * std_diff_population / np.sqrt(max(len(diff), 1))

    return {
        "mean_diff_b_minus_a": mean_diff,
        "cohens_dz": dz,
        "ci95_low": float(mean_diff - half),
        "ci95_high": float(mean_diff + half),
    }


def safe_get(d, keys, default=np.nan):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def save_table(df, filename):
    path = ANALYSIS_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)


def save_plot(filename):
    path = ANALYSIS_DIR / filename
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved:", path)

### Full scratch results: baselines, stable sweep, data-size sweep

In [ ]:
full_manifest = load_json(full_root / "run_manifest.json")

print("FULL SCRATCH MANIFEST")
print("=" * 100)
for k, v in full_manifest.items():
    print(f"{k:<28}: {v}")


baseline_df = pd.DataFrame()
baseline_path = full_root / "baseline_results.json"

if baseline_path.exists():
    baselines = load_json(baseline_path)
    baseline_df = pd.DataFrame(baselines)

    keep_cols = [
        "model_name", "num_parameters", "train_acc", "test_acc",
        "train_error_01", "test_error_01", "generalization_gap_01",
        "test_ece", "test_nll", "test_brier"
    ]
    keep_cols = [c for c in keep_cols if c in baseline_df.columns]
    baseline_df = baseline_df[keep_cols]

    print("\nDBPEDIA BASELINES")
    display(baseline_df)
    save_table(baseline_df, "dbpedia_baselines.csv")


stable_df = pd.DataFrame()
stable_shift_df = pd.DataFrame()

stable_path = full_root / "stable_sweep" / "stable_sweep_summary.json"

if stable_path.exists():
    stable = load_json(stable_path)

    stable_rows = []
    shift_rows = []

    for s in stable:
        stable_rows.append({
            "dataset_name": s["dataset_name"],
            "model_name": s["model_name"],
            "condition": s["condition"],
            "num_parameters": s["num_parameters"],
            "seeds": s["seeds"],
            "test_acc_mean": s["test_acc_stats"]["mean"],
            "test_acc_std": s["test_acc_stats"]["std"],
            "test_acc_ci95_low": s["test_acc_stats"]["ci95_low"],
            "test_acc_ci95_high": s["test_acc_stats"]["ci95_high"],
            "gap_mean": s["gap_stats"]["mean"],
            "gap_std": s["gap_stats"]["std"],
            "ece_mean": s["ece_stats"]["mean"],
            "ece_scaled_mean": s["ece_scaled_stats"]["mean"],
            "nll_mean": s["nll_stats"]["mean"],
            "nll_scaled_mean": s["nll_scaled_stats"]["mean"],
            "brier_mean": s["brier_stats"]["mean"],
            "brier_scaled_mean": s["brier_scaled_stats"]["mean"],
            "scheduler_trace_steps_mean": safe_get(s, ["scheduler_trace_steps_stats", "mean"]),
        })

        if "shift_summary" in s:
            clean_acc = s["test_acc_stats"]["mean"]
            for shift_name, vals in s["shift_summary"].items():
                shift_rows.append({
                    "model_name": s["model_name"],
                    "condition": s["condition"],
                    "shift_name": shift_name,
                    "clean_acc": clean_acc,
                    "shift_acc": vals["test_acc_stats"]["mean"],
                    "acc_drop": clean_acc - vals["test_acc_stats"]["mean"],
                    "ece": vals["ece_stats"]["mean"],
                    "nll": vals["nll_stats"]["mean"],
                    "brier": vals["brier_stats"]["mean"],
                })

    stable_df = pd.DataFrame(stable_rows).sort_values("num_parameters").reset_index(drop=True)
    stable_shift_df = pd.DataFrame(shift_rows)

    print("\nDBPEDIA STABLE SWEEP")
    display(stable_df)
    save_table(stable_df, "dbpedia_stable_sweep.csv")

    if not stable_shift_df.empty:
        print("\nDBPEDIA STABLE-SCHEDULE SHIFT SUMMARY")
        display(stable_shift_df)
        save_table(stable_shift_df, "dbpedia_stable_shift_summary.csv")


data_size_df = pd.DataFrame()
data_size_path = full_root / "data_size_sweep" / "data_size_sweep_summary.json"

if data_size_path.exists():
    data_size = load_json(data_size_path)
    rows = []

    for block in data_size:
        for frac in block["fractions"]:
            rows.append({
                "model_name": block["model_name"],
                "train_fraction": frac["train_fraction"],
                "condition": frac["condition"],
                "num_parameters": frac["num_parameters"],
                "seeds": frac["seeds"],
                "test_acc_mean": frac["test_acc_stats"]["mean"],
                "test_acc_std": frac["test_acc_stats"]["std"],
                "test_acc_ci95_low": frac["test_acc_stats"]["ci95_low"],
                "test_acc_ci95_high": frac["test_acc_stats"]["ci95_high"],
                "gap_mean": frac["gap_stats"]["mean"],
                "gap_std": frac["gap_stats"]["std"],
                "ece_mean": frac["ece_stats"]["mean"],
                "ece_scaled_mean": frac["ece_scaled_stats"]["mean"],
                "nll_mean": frac["nll_stats"]["mean"],
                "nll_scaled_mean": frac["nll_scaled_stats"]["mean"],
            })

    data_size_df = pd.DataFrame(rows).sort_values(["model_name", "train_fraction"]).reset_index(drop=True)

    print("\nDBPEDIA DATA-SIZE SWEEP")
    display(data_size_df)
    save_table(data_size_df, "dbpedia_data_size_sweep.csv")

### Full scratch plots

In [ ]:
if not stable_df.empty:
    plt.figure(figsize=(8, 5))

    stable_plot = stable_df.sort_values("num_parameters")
    plt.errorbar(
        stable_plot["num_parameters"],
        stable_plot["test_acc_mean"],
        yerr=stable_plot["test_acc_std"],
        fmt="o-",
        capsize=4,
        linewidth=2,
    )

    for _, row in stable_plot.iterrows():
        plt.annotate(
            row["model_name"],
            (row["num_parameters"], row["test_acc_mean"]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
        )

    plt.xscale("log")
    plt.title("DBPedia stable sweep: test accuracy vs parameters")
    plt.xlabel("Parameters")
    plt.ylabel("Test accuracy")
    plt.grid(alpha=0.3)

    save_plot("dbpedia_stable_accuracy_vs_params.png")
    plt.show()


if not stable_df.empty:
    plt.figure(figsize=(8, 5))

    stable_plot = stable_df.sort_values("num_parameters")
    plt.errorbar(
        stable_plot["num_parameters"],
        stable_plot["gap_mean"],
        yerr=stable_plot["gap_std"],
        fmt="o-",
        capsize=4,
        linewidth=2,
    )

    for _, row in stable_plot.iterrows():
        plt.annotate(
            row["model_name"],
            (row["num_parameters"], row["gap_mean"]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
        )

    plt.xscale("log")
    plt.title("DBPedia stable sweep: generalization gap vs parameters")
    plt.xlabel("Parameters")
    plt.ylabel("Generalization gap")
    plt.grid(alpha=0.3)

    save_plot("dbpedia_stable_gap_vs_params.png")
    plt.show()


if not data_size_df.empty:
    plt.figure(figsize=(8, 5))

    for model_name in sorted(data_size_df["model_name"].unique()):
        sub = data_size_df[data_size_df["model_name"] == model_name].sort_values("train_fraction")
        plt.plot(
            sub["train_fraction"],
            sub["test_acc_mean"],
            marker="o",
            linewidth=2,
            label=model_name,
        )

    plt.title("DBPedia data-size sweep")
    plt.xlabel("Train fraction")
    plt.ylabel("Test accuracy")
    plt.grid(alpha=0.3)
    plt.legend()

    save_plot("dbpedia_data_size_sweep.png")
    plt.show()

### Ablation + shift analysis

In [ ]:
ablation_manifest = load_json(ablation_root / "run_manifest.json")

print("ABLATION + SHIFT MANIFEST")
print("=" * 100)
for k, v in ablation_manifest.items():
    print(f"{k:<28}: {v}")

ablation_summary = load_json(ablation_root / "ablation" / "ablation_summary.json")

ablation_rows = []
per_seed_rows = []
shift_rows = []

for block in ablation_summary:
    model_name = block["model_name"]

    for cond in block["conditions"]:
        condition = cond["condition"]

        ablation_rows.append({
            "model_name": model_name,
            "condition": condition,
            "num_parameters": cond["num_parameters"],
            "seeds": cond["seeds"],
            "test_acc_mean": cond["test_acc_stats"]["mean"],
            "test_acc_std": cond["test_acc_stats"]["std"],
            "test_acc_ci95_low": cond["test_acc_stats"]["ci95_low"],
            "test_acc_ci95_high": cond["test_acc_stats"]["ci95_high"],
            "gap_mean": cond["gap_stats"]["mean"],
            "gap_std": cond["gap_stats"]["std"],
            "ece_mean": cond["ece_stats"]["mean"],
            "ece_scaled_mean": cond["ece_scaled_stats"]["mean"],
            "nll_mean": cond["nll_stats"]["mean"],
            "nll_scaled_mean": cond["nll_scaled_stats"]["mean"],
            "brier_mean": cond["brier_stats"]["mean"],
            "brier_scaled_mean": cond["brier_scaled_stats"]["mean"],
            "scheduler_trace_steps_mean": safe_get(cond, ["scheduler_trace_steps_stats", "mean"]),
        })

        for r in cond["per_seed_results"]:
            per_seed_rows.append({
                "model_name": model_name,
                "condition": condition,
                "seed": int(r["seed"]),
                "test_acc": float(r["test_acc"]),
                "test_ece": float(r["test_ece"]),
                "test_nll": float(r["test_nll"]),
                "test_brier": float(r["test_brier"]),
                "test_ece_scaled": float(r["test_ece_scaled"]),
                "test_nll_scaled": float(r["test_nll_scaled"]),
                "test_brier_scaled": float(r["test_brier_scaled"]),
                "generalization_gap_01": float(r["generalization_gap_01"]),
                "scheduler_trace_steps": int(r.get("scheduler_trace_steps", 0)),
                "scheduler_trace_file": r.get("scheduler_trace_file", ""),
            })

        if "shift_summary" in cond:
            clean_acc = cond["test_acc_stats"]["mean"]
            for shift_name, vals in cond["shift_summary"].items():
                shift_rows.append({
                    "model_name": model_name,
                    "condition": condition,
                    "shift_name": shift_name,
                    "clean_acc": clean_acc,
                    "shift_acc": vals["test_acc_stats"]["mean"],
                    "acc_drop": clean_acc - vals["test_acc_stats"]["mean"],
                    "ece": vals["ece_stats"]["mean"],
                    "nll": vals["nll_stats"]["mean"],
                    "brier": vals["brier_stats"]["mean"],
                })

ablation_df = pd.DataFrame(ablation_rows).sort_values(["model_name", "condition"]).reset_index(drop=True)
per_seed_df = pd.DataFrame(per_seed_rows).sort_values(["model_name", "seed", "condition"]).reset_index(drop=True)
shift_df = pd.DataFrame(shift_rows).sort_values(["model_name", "condition", "shift_name"]).reset_index(drop=True)

print("\nDBPEDIA ABLATION SUMMARY")
display(ablation_df)

print("\nDBPEDIA PER-SEED ABLATION TABLE")
display(per_seed_df)

print("\nDBPEDIA SHIFT SUMMARY FOR ALL CONDITIONS")
display(shift_df)

save_table(ablation_df, "dbpedia_ablation_summary.csv")
save_table(per_seed_df, "dbpedia_ablation_per_seed.csv")
save_table(shift_df, "dbpedia_shift_all_conditions.csv")

### Effect sizes + confidence intervals

In [ ]:
comparisons = [
    ("stable_schedule", "legacy_warmup_fixed_order"),
    ("stable_schedule", "legacy_buggy_warmup"),
    ("legacy_warmup_fixed_order", "legacy_buggy_warmup"),
]

metrics = [
    "test_acc",
    "test_ece",
    "test_nll",
    "generalization_gap_01",
]

effect_rows = []

for model_name in sorted(per_seed_df["model_name"].unique()):
    sub_model = per_seed_df[per_seed_df["model_name"] == model_name].copy()

    for cond_a, cond_b in comparisons:
        a_df = sub_model[sub_model["condition"] == cond_a].copy()
        b_df = sub_model[sub_model["condition"] == cond_b].copy()

        common_seeds = sorted(set(a_df["seed"]) & set(b_df["seed"]))
        if len(common_seeds) == 0:
            continue

        a_df = a_df[a_df["seed"].isin(common_seeds)].sort_values("seed")
        b_df = b_df[b_df["seed"].isin(common_seeds)].sort_values("seed")

        for metric in metrics:
            stats = paired_effect_summary(a_df[metric].values, b_df[metric].values)

            effect_rows.append({
                "model_name": model_name,
                "comparison": f"{cond_a} -> {cond_b}",
                "metric": metric,
                "n_seeds": len(common_seeds),
                "mean_a": float(a_df[metric].mean()),
                "mean_b": float(b_df[metric].mean()),
                "mean_diff_b_minus_a": stats["mean_diff_b_minus_a"],
                "cohens_dz": stats["cohens_dz"],
                "ci95_low": stats["ci95_low"],
                "ci95_high": stats["ci95_high"],
            })

effect_df = pd.DataFrame(effect_rows)

print("DBPEDIA EFFECT SIZE + CI TABLE")
display(effect_df)

save_table(effect_df, "dbpedia_effect_sizes_ci.csv")

### Paired per-seed plots

In [ ]:
plot_metrics = [
    ("test_acc", "Accuracy"),
    ("test_ece", "ECE"),
    ("test_nll", "NLL"),
]

condition_order = [
    "stable_schedule",
    "legacy_warmup_fixed_order",
    "legacy_buggy_warmup",
]

for model_name in sorted(per_seed_df["model_name"].unique()):
    sub_model = per_seed_df[per_seed_df["model_name"] == model_name].copy()
    seeds = sorted(sub_model["seed"].unique())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"DBPedia paired per-seed comparison — {model_name}", fontsize=13, fontweight="bold")

    for ax, (metric, ylabel) in zip(axes, plot_metrics):
        pivot = sub_model.pivot(index="seed", columns="condition", values=metric).reindex(index=seeds)
        x = np.arange(len(condition_order))

        for seed in seeds:
            y = [pivot.loc[seed, c] if c in pivot.columns else np.nan for c in condition_order]
            ax.plot(x, y, marker="o", linewidth=2, label=f"seed {seed}")

        ax.set_xticks(x)
        ax.set_xticklabels(condition_order, rotation=20, ha="right")
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
        ax.grid(alpha=0.3)

    axes[0].legend()
    plt.tight_layout()

    save_plot(f"dbpedia_paired_per_seed_{model_name}.png")
    plt.show()

### Scheduler trace analysis

In [ ]:
trace_rows = []
first_steps_rows = []

trace_dir = ablation_root / "scheduler_traces"

if not trace_dir.exists():
    raise FileNotFoundError(f"Scheduler trace folder not found: {trace_dir}")

for _, row in per_seed_df.iterrows():
    rel_file = row["scheduler_trace_file"]
    trace_path = ablation_root / rel_file

    if not trace_path.exists():
        possible = list(trace_dir.glob(f"*{row['model_name']}*{row['condition']}*seed{row['seed']}*.json"))
        if possible:
            trace_path = possible[0]

    if not trace_path.exists():
        trace_rows.append({
            "model_name": row["model_name"],
            "condition": row["condition"],
            "seed": row["seed"],
            "trace_found": False,
        })
        continue

    trace = load_json(trace_path)

    if len(trace) == 0:
        trace_rows.append({
            "model_name": row["model_name"],
            "condition": row["condition"],
            "seed": row["seed"],
            "trace_found": True,
            "num_steps": 0,
        })
        continue

    lr_used = np.array([x["lr_used"] for x in trace], dtype=np.float64)
    lr_before = np.array([x["lr_before_step"] for x in trace], dtype=np.float64)
    lr_after = np.array([x["lr_after_step"] for x in trace], dtype=np.float64)

    trace_rows.append({
        "model_name": row["model_name"],
        "condition": row["condition"],
        "seed": row["seed"],
        "trace_found": True,
        "num_steps": len(trace),
        "first_lr_before_step": float(lr_before[0]),
        "first_lr_used": float(lr_used[0]),
        "first_lr_after_step": float(lr_after[0]),
        "max_lr_used_first50": float(np.max(lr_used[:min(50, len(lr_used))])),
        "max_lr_used_all": float(np.max(lr_used)),
        "last_lr_used": float(lr_used[-1]),
        "trace_file": str(trace_path),
    })

    for x in trace[:5]:
        first_steps_rows.append({
            "model_name": row["model_name"],
            "condition": row["condition"],
            "seed": row["seed"],
            "step": x["step"],
            "lr_before_step": x["lr_before_step"],
            "lr_used": x["lr_used"],
            "lr_after_step": x["lr_after_step"],
        })

scheduler_summary_df = pd.DataFrame(trace_rows).sort_values(["model_name", "condition", "seed"]).reset_index(drop=True)
scheduler_first_steps_df = pd.DataFrame(first_steps_rows).sort_values(["model_name", "condition", "seed", "step"]).reset_index(drop=True)

print("DBPEDIA SCHEDULER TRACE SUMMARY")
display(scheduler_summary_df)

print("\nDBPEDIA FIRST 5 SCHEDULER STEPS")
display(scheduler_first_steps_df)

save_table(scheduler_summary_df, "dbpedia_scheduler_trace_summary.csv")
save_table(scheduler_first_steps_df, "dbpedia_scheduler_first_5_steps.csv")

### Scheduler trace plots

In [ ]:
for model_name in sorted(per_seed_df["model_name"].unique()):
    plt.figure(figsize=(8, 5))

    for condition in condition_order:
        sub = scheduler_summary_df[
            (scheduler_summary_df["model_name"] == model_name) &
            (scheduler_summary_df["condition"] == condition) &
            (scheduler_summary_df["seed"] == 42) &
            (scheduler_summary_df["trace_found"] == True)
        ]

        if sub.empty:
            continue

        trace_path = Path(sub.iloc[0]["trace_file"])
        trace = load_json(trace_path)
        trace_df = pd.DataFrame(trace).head(100)

        plt.plot(
            trace_df["step"],
            trace_df["lr_used"],
            marker="o",
            linewidth=2,
            label=condition,
        )

    plt.title(f"DBPedia scheduler trace, first 100 steps — {model_name}, seed 42")
    plt.xlabel("Step")
    plt.ylabel("LR used")
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.legend()

    save_plot(f"dbpedia_scheduler_trace_first100_{model_name}.png")
    plt.show()

### Early diagnostics: load all runs

In [ ]:
earlydiag_manifest = load_json(earlydiag_root / "run_manifest.json")
all_runs = load_json(earlydiag_root / "all_runs.json")

print("EARLY DIAGNOSTICS MANIFEST")
print("=" * 100)
for k, v in earlydiag_manifest.items():
    print(f"{k:<28}: {v}")

print("\nNumber of early diagnostic runs:", len(all_runs))

### Early diagnostics: 50 / 100 / 300 window summaries

In [ ]:
def build_window_summary_from_logs(logs, step_limit):
    sub = logs[:min(step_limit, len(logs))]
    if len(sub) == 0:
        return {}

    layer_names = sorted({
        layer_name
        for x in sub
        for layer_name in x.get("layer_update_to_param_ratio", {}).keys()
    })

    layer_max = {}
    for layer_name in layer_names:
        vals = [
            x.get("layer_update_to_param_ratio", {}).get(layer_name, np.nan)
            for x in sub
        ]
        vals = [v for v in vals if not np.isnan(v)]
        if vals:
            layer_max[layer_name] = float(max(vals))

    repr_rows = [x for x in sub if x.get("repr_drift_l2") is not None]

    emb_vals = [
        x.get("layer_update_to_param_ratio", {}).get("embedding", np.nan)
        for x in sub
    ]
    emb_vals = [v for v in emb_vals if not np.isnan(v)]

    cls_vals = [
        x.get("layer_update_to_param_ratio", {}).get("classifier", np.nan)
        for x in sub
    ]
    cls_vals = [v for v in cls_vals if not np.isnan(v)]

    return {
        "steps_included": int(len(sub)),
        "loss_step1": float(sub[0]["loss"]),
        "max_loss": float(max(x["loss"] for x in sub)),
        "max_grad_norm": float(max(x["grad_norm"] for x in sub)),
        "max_update_ratio": float(max(x["update_to_param_ratio"] for x in sub)),
        "max_embedding_update_ratio": float(max(emb_vals)) if emb_vals else np.nan,
        "max_classifier_update_ratio": float(max(cls_vals)) if cls_vals else np.nan,
        "mean_confidence_first10_within_window": float(np.mean([x["mean_confidence"] for x in sub[:min(10, len(sub))]])),
        "last_repr_drift_l2": float(repr_rows[-1]["repr_drift_l2"]) if repr_rows else np.nan,
        "last_repr_cos_to_init": float(repr_rows[-1]["repr_cos_to_init"]) if repr_rows else np.nan,
        "last_lr_before_step": float(sub[-1]["lr_before_step"]),
        "last_lr_used": float(sub[-1]["lr_used"]),
        "last_lr_after_step": float(sub[-1]["lr_after_step"]),
        "max_layer_update_ratio": layer_max,
    }


window_rows = []
layer_rows = []

for run in all_runs:
    logs = run["logs"]

    for window in [50, 100, 300]:
        # Always rebuild from logs so layerwise values are present.
        summ = build_window_summary_from_logs(logs, window)

        if not summ:
            continue

        window_rows.append({
            "model_name": run["model_name"],
            "condition": run["condition"],
            "seed": int(run["seed"]),
            "window": window,
            "steps_included": summ.get("steps_included", np.nan),
            "loss_step1": summ.get("loss_step1", np.nan),
            "max_loss": summ.get("max_loss", np.nan),
            "max_grad_norm": summ.get("max_grad_norm", np.nan),
            "max_update_ratio": summ.get("max_update_ratio", np.nan),
            "max_embedding_update_ratio": summ.get("max_embedding_update_ratio", np.nan),
            "max_classifier_update_ratio": summ.get("max_classifier_update_ratio", np.nan),
            "mean_confidence_first10_within_window": summ.get("mean_confidence_first10_within_window", np.nan),
            "last_repr_drift_l2": summ.get("last_repr_drift_l2", np.nan),
            "last_repr_cos_to_init": summ.get("last_repr_cos_to_init", np.nan),
            "last_lr_before_step": summ.get("last_lr_before_step", np.nan),
            "last_lr_used": summ.get("last_lr_used", np.nan),
            "last_lr_after_step": summ.get("last_lr_after_step", np.nan),
        })

        layer_dict = summ.get("max_layer_update_ratio", {})
        for layer_name, val in layer_dict.items():
            layer_rows.append({
                "model_name": run["model_name"],
                "condition": run["condition"],
                "seed": int(run["seed"]),
                "window": window,
                "layer_name": layer_name,
                "max_layer_update_ratio": val,
            })

window_df = pd.DataFrame(window_rows)

if not window_df.empty:
    window_df = window_df.sort_values(
        ["model_name", "condition", "seed", "window"]
    ).reset_index(drop=True)

layer_df = pd.DataFrame(layer_rows)

if not layer_df.empty:
    layer_df = layer_df.sort_values(
        ["model_name", "condition", "seed", "window", "layer_name"]
    ).reset_index(drop=True)
else:
    layer_df = pd.DataFrame(columns=[
        "model_name",
        "condition",
        "seed",
        "window",
        "layer_name",
        "max_layer_update_ratio",
    ])

print("DBPEDIA EARLY WINDOW SUMMARY")
display(window_df)

print("\nDBPEDIA LAYERWISE WINDOW SUMMARY")
display(layer_df)

save_table(window_df, "dbpedia_early_window_summary_50_100_300.csv")
save_table(layer_df, "dbpedia_layerwise_window_summary_50_100_300.csv")

### Early diagnostics aggregate CI table

In [ ]:
agg_rows = []

early_metrics = [
    "max_loss",
    "max_grad_norm",
    "max_update_ratio",
    "max_embedding_update_ratio",
    "max_classifier_update_ratio",
    "mean_confidence_first10_within_window",
    "last_repr_drift_l2",
    "last_repr_cos_to_init",
    "last_lr_used",
]

for (model_name, condition, window), sub in window_df.groupby(["model_name", "condition", "window"]):
    for metric in early_metrics:
        ci = mean_ci95_from_values(sub[metric].values)
        agg_rows.append({
            "model_name": model_name,
            "condition": condition,
            "window": window,
            "metric": metric,
            "mean": ci["mean"],
            "std": ci["std"],
            "ci95_low": ci["ci95_low"],
            "ci95_high": ci["ci95_high"],
            "n": ci["n"],
        })

window_agg_df = pd.DataFrame(agg_rows)

print("DBPEDIA EARLY WINDOW AGGREGATE CI TABLE")
display(window_agg_df)

save_table(window_agg_df, "dbpedia_early_window_aggregate_ci.csv")

### Early diagnostics plots

In [ ]:
for model_name in sorted(window_df["model_name"].unique()):
    sub = window_df[window_df["model_name"] == model_name].copy()

    plt.figure(figsize=(8, 5))

    for condition in condition_order:
        s = (
            sub[sub["condition"] == condition]
            .groupby("window")["max_update_ratio"]
            .mean()
            .reset_index()
        )

        if s.empty:
            continue

        plt.plot(
            s["window"],
            s["max_update_ratio"],
            marker="o",
            linewidth=2,
            label=condition,
        )

    plt.title(f"DBPedia early max update/parameter ratio — {model_name}")
    plt.xlabel("Window")
    plt.ylabel("Max update / parameter ratio")
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.legend()

    save_plot(f"dbpedia_early_max_update_ratio_{model_name}.png")
    plt.show()


for model_name in sorted(window_df["model_name"].unique()):
    sub = window_df[window_df["model_name"] == model_name].copy()

    plt.figure(figsize=(8, 5))

    for condition in condition_order:
        s = (
            sub[sub["condition"] == condition]
            .groupby("window")["max_grad_norm"]
            .mean()
            .reset_index()
        )

        if s.empty:
            continue

        plt.plot(
            s["window"],
            s["max_grad_norm"],
            marker="o",
            linewidth=2,
            label=condition,
        )

    plt.title(f"DBPedia early max gradient norm — {model_name}")
    plt.xlabel("Window")
    plt.ylabel("Max gradient norm")
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.legend()

    save_plot(f"dbpedia_early_max_grad_norm_{model_name}.png")
    plt.show()

### Early diagnostic traces for seed 42

In [ ]:
for model_name in sorted(window_df["model_name"].unique()):
    subset_runs = [
        r for r in all_runs
        if r["model_name"] == model_name and int(r["seed"]) == 42
    ]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f"DBPedia early diagnostics — {model_name}, seed 42", fontsize=13, fontweight="bold")

    for run in subset_runs:
        condition = run["condition"]
        log_df = pd.DataFrame(run["logs"])

        axes[0, 0].plot(log_df["step"], log_df["loss"], label=condition, linewidth=2)
        axes[0, 1].plot(log_df["step"], log_df["lr_used"], label=condition, linewidth=2)
        axes[0, 2].plot(log_df["step"], log_df["grad_norm"], label=condition, linewidth=2)
        axes[1, 0].plot(log_df["step"], log_df["update_to_param_ratio"], label=condition, linewidth=2)

        rep_df = log_df[log_df["repr_drift_l2"].notna()].copy()
        axes[1, 1].plot(rep_df["step"], rep_df["repr_drift_l2"], label=condition, linewidth=2)
        axes[1, 2].plot(rep_df["step"], rep_df["repr_cos_to_init"], label=condition, linewidth=2)

    axes[0, 0].set_title("Loss")
    axes[0, 0].set_yscale("log")

    axes[0, 1].set_title("LR used")
    axes[0, 1].set_yscale("log")

    axes[0, 2].set_title("Gradient norm")
    axes[0, 2].set_yscale("log")

    axes[1, 0].set_title("Update / parameter ratio")
    axes[1, 0].set_yscale("log")

    axes[1, 1].set_title("Representation drift L2")
    axes[1, 1].set_yscale("log")

    axes[1, 2].set_title("Cosine similarity to initialization")

    for ax in axes.ravel():
        ax.set_xlabel("Step")
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout()

    save_plot(f"dbpedia_earlydiag_trace_seed42_{model_name}.png")
    plt.show()

### Initial-LR causal sweep analysis

In [ ]:
initial_lr_manifest = load_json(initial_lr_root / "run_manifest.json")
initial_lr_summary = load_json(initial_lr_root / "initial_lr_sweep_summary.json")

print("INITIAL-LR SWEEP MANIFEST")
print("=" * 100)
for k, v in initial_lr_manifest.items():
    print(f"{k:<28}: {v}")

initial_lr_rows = []
initial_lr_per_seed_rows = []

for block in initial_lr_summary:
    model_name = block["model_name"]

    for lr_block in block["initial_lr_blocks"]:
        initial_lr = lr_block["initial_lr"]

        initial_lr_rows.append({
            "model_name": model_name,
            "condition": lr_block["condition"],
            "initial_lr": initial_lr,
            "num_parameters": lr_block["num_parameters"],
            "seeds": lr_block["seeds"],
            "test_acc_mean": lr_block["test_acc_stats"]["mean"],
            "test_acc_std": lr_block["test_acc_stats"]["std"],
            "test_acc_ci95_low": lr_block["test_acc_stats"]["ci95_low"],
            "test_acc_ci95_high": lr_block["test_acc_stats"]["ci95_high"],
            "gap_mean": lr_block["gap_stats"]["mean"],
            "ece_mean": lr_block["ece_stats"]["mean"],
            "ece_scaled_mean": lr_block["ece_scaled_stats"]["mean"],
            "nll_mean": lr_block["nll_stats"]["mean"],
            "nll_scaled_mean": lr_block["nll_scaled_stats"]["mean"],
            "scheduler_trace_steps_mean": safe_get(lr_block, ["scheduler_trace_steps_stats", "mean"]),
        })

        for r in lr_block["per_seed_results"]:
            initial_lr_per_seed_rows.append({
                "model_name": model_name,
                "condition": r["condition"],
                "initial_lr": float(r["initial_lr"]),
                "seed": int(r["seed"]),
                "test_acc": float(r["test_acc"]),
                "test_ece": float(r["test_ece"]),
                "test_nll": float(r["test_nll"]),
                "generalization_gap_01": float(r["generalization_gap_01"]),
                "scheduler_trace_steps": int(r.get("scheduler_trace_steps", 0)),
                "scheduler_trace_file": r.get("scheduler_trace_file", ""),
            })

initial_lr_df = pd.DataFrame(initial_lr_rows).sort_values(["model_name", "initial_lr"], ascending=[True, False]).reset_index(drop=True)
initial_lr_per_seed_df = pd.DataFrame(initial_lr_per_seed_rows).sort_values(["model_name", "initial_lr", "seed"], ascending=[True, False, True]).reset_index(drop=True)

print("\nDBPEDIA INITIAL-LR SWEEP SUMMARY")
display(initial_lr_df)

print("\nDBPEDIA INITIAL-LR PER-SEED TABLE")
display(initial_lr_per_seed_df)

save_table(initial_lr_df, "dbpedia_initial_lr_sweep_summary.csv")
save_table(initial_lr_per_seed_df, "dbpedia_initial_lr_sweep_per_seed.csv")

### Initial-LR sweep plots

In [ ]:
for model_name in sorted(initial_lr_df["model_name"].unique()):
    sub = initial_lr_df[initial_lr_df["model_name"] == model_name].sort_values("initial_lr")

    plt.figure(figsize=(8, 5))

    plt.errorbar(
        sub["initial_lr"],
        sub["test_acc_mean"],
        yerr=sub["test_acc_std"],
        marker="o",
        linewidth=2,
        capsize=4,
    )

    plt.xscale("log")
    plt.title(f"DBPedia initial-LR sweep — {model_name}")
    plt.xlabel("Initial LR used by buggy warmup")
    plt.ylabel("Test accuracy")
    plt.grid(alpha=0.3)

    save_plot(f"dbpedia_initial_lr_sweep_accuracy_{model_name}.png")
    plt.show()


for model_name in sorted(initial_lr_df["model_name"].unique()):
    sub = initial_lr_df[initial_lr_df["model_name"] == model_name].sort_values("initial_lr")

    plt.figure(figsize=(8, 5))

    plt.plot(
        sub["initial_lr"],
        sub["nll_mean"],
        marker="o",
        linewidth=2,
    )

    plt.xscale("log")
    plt.yscale("log")
    plt.title(f"DBPedia initial-LR sweep — NLL — {model_name}")
    plt.xlabel("Initial LR used by buggy warmup")
    plt.ylabel("NLL")
    plt.grid(alpha=0.3)

    save_plot(f"dbpedia_initial_lr_sweep_nll_{model_name}.png")
    plt.show()

### Initial-LR first-step scheduler check

In [ ]:
initial_lr_trace_rows = []
trace_dir = initial_lr_root / "scheduler_traces"

for _, row in initial_lr_per_seed_df.iterrows():
    rel_file = row["scheduler_trace_file"]
    trace_path = initial_lr_root / rel_file

    if not trace_path.exists():
        possible = list(trace_dir.glob(f"*{row['model_name']}*initlr*seed{row['seed']}*.json"))
        if possible:
            trace_path = possible[0]

    if not trace_path.exists():
        initial_lr_trace_rows.append({
            "model_name": row["model_name"],
            "initial_lr": row["initial_lr"],
            "seed": row["seed"],
            "trace_found": False,
        })
        continue

    trace = load_json(trace_path)
    first = trace[0] if len(trace) > 0 else {}

    initial_lr_trace_rows.append({
        "model_name": row["model_name"],
        "initial_lr": row["initial_lr"],
        "seed": row["seed"],
        "trace_found": True,
        "first_lr_before_step": first.get("lr_before_step", np.nan),
        "first_lr_used": first.get("lr_used", np.nan),
        "first_lr_after_step": first.get("lr_after_step", np.nan),
        "num_steps": len(trace),
        "trace_file": str(trace_path),
    })

initial_lr_trace_df = pd.DataFrame(initial_lr_trace_rows).sort_values(
    ["model_name", "initial_lr", "seed"],
    ascending=[True, False, True]
)

print("DBPEDIA INITIAL-LR FIRST-STEP TRACE CHECK")
display(initial_lr_trace_df)

save_table(initial_lr_trace_df, "dbpedia_initial_lr_first_step_trace_check.csv")

### Compact final interpretation numbers

In [ ]:
def get_effect(model_name, comparison, metric):
    sub = effect_df[
        (effect_df["model_name"] == model_name) &
        (effect_df["comparison"] == comparison) &
        (effect_df["metric"] == metric)
    ]
    if sub.empty:
        return None
    return sub.iloc[0].to_dict()


print("DBPEDIA COMPACT INTERPRETATION")
print("=" * 100)

for model_name in sorted(per_seed_df["model_name"].unique()):
    print(f"\nMODEL: {model_name}")

    e_acc = get_effect(model_name, "stable_schedule -> legacy_buggy_warmup", "test_acc")
    e_ece = get_effect(model_name, "stable_schedule -> legacy_buggy_warmup", "test_ece")
    e_nll = get_effect(model_name, "stable_schedule -> legacy_buggy_warmup", "test_nll")

    if e_acc is not None:
        print(
            f"Stable -> buggy accuracy diff: {e_acc['mean_diff_b_minus_a']:.4f} "
            f"[{e_acc['ci95_low']:.4f}, {e_acc['ci95_high']:.4f}], dz={e_acc['cohens_dz']:.3f}"
        )

    if e_ece is not None:
        print(
            f"Stable -> buggy ECE diff     : {e_ece['mean_diff_b_minus_a']:.4f} "
            f"[{e_ece['ci95_low']:.4f}, {e_ece['ci95_high']:.4f}], dz={e_ece['cohens_dz']:.3f}"
        )

    if e_nll is not None:
        print(
            f"Stable -> buggy NLL diff     : {e_nll['mean_diff_b_minus_a']:.4f} "
            f"[{e_nll['ci95_low']:.4f}, {e_nll['ci95_high']:.4f}], dz={e_nll['cohens_dz']:.3f}"
        )

print("\nInitial-LR sweep means:")
display(initial_lr_df[["model_name", "initial_lr", "test_acc_mean", "test_acc_std", "ece_mean", "nll_mean"]])

### Save all DBPedia analysis artifacts as zip

In [ ]:
zip_path = Path("/content/dbpedia_final_analysis_artifacts.zip")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in ANALYSIS_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(ANALYSIS_DIR)))

print("Created:", zip_path)
files.download(str(zip_path))

In [ ]:
from pathlib import Path

loader_path = Path("transformer_project/data/data_loader_scratch.py")

if not loader_path.exists():
    raise FileNotFoundError("Missing transformer_project/data/data_loader_scratch.py")

code = loader_path.read_text()

code = code.replace('"hf_name": "dbpedia_14"', '"hf_name": "fancyzhx/dbpedia_14"')
code = code.replace("'hf_name': 'dbpedia_14'", "'hf_name': 'fancyzhx/dbpedia_14'")

# also make AG News robust, but this will not affect DBPedia
code = code.replace('"hf_name": "ag_news"', '"hf_name": "fancyzhx/ag_news"')
code = code.replace("'hf_name': 'ag_news'", "'hf_name': 'fancyzhx/ag_news'")

loader_path.write_text(code)

print("Patched dataset names in:", loader_path)
print("DBPedia should now use fancyzhx/dbpedia_14")

### Create the DBPedia Warmup Safety Gate script

In [ ]:
%%writefile transformer_project/run_warmup_safety_gate_dbpedia.py
"""
DBPedia Warmup Safety Gate experiment with calibrated update-ratio threshold.

Threshold rule:
- Run stable_with_guard and fixed_order_with_guard first.
- Compute max early update/parameter ratio from only those safe runs.
- Set threshold = 10 × max_safe_update_ratio.
- Manifest records threshold_source = calibrated_10x_safe_max.
"""

import os
import sys
import json
import copy
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT_ROOT = Path(__file__).resolve().parent
sys.path.append(str(PROJECT_ROOT))

from data.data_loader_scratch import load_text_dataset, get_dataloaders
from models.transformer import TransformerClassifier, create_model_configs
from utils.metrics_scratch import (
    expected_calibration_error,
    multiclass_nll,
    multiclass_brier,
    mean_ci95,
)

RUN_MODE = "quick"      # quick test: seed 42, 1 epoch, 1% training data
# RUN_MODE = "full"     # full final run: seeds 42,123,456, 30 epochs, full data

DATASET_NAME = "dbpedia_14"
RUN_TAG = "dbpedia_warmup_safety_gate_v2_calibrated"

if RUN_MODE == "quick":
    SEEDS = [42]
    MODEL_NAMES = ["medium-4", "large-6"]
    NUM_EPOCHS = 1
    TRAIN_FRACTION = 0.01
elif RUN_MODE == "full":
    SEEDS = [42, 123, 456]
    MODEL_NAMES = ["medium-4", "large-6"]
    NUM_EPOCHS = 30
    TRAIN_FRACTION = 1.0
else:
    raise ValueError("RUN_MODE must be 'quick' or 'full'")

CALIBRATION_CONDITIONS = [
    "stable_with_guard",
    "fixed_order_with_guard",
]

POST_CALIBRATION_CONDITIONS = [
    "buggy_no_guard",
    "buggy_failfast_guard",
    "buggy_rescue_guard",
    "single_bad_first_step_then_stable",
]

CONDITIONS = [
    "buggy_no_guard",
    "buggy_failfast_guard",
    "buggy_rescue_guard",
    "stable_with_guard",
    "fixed_order_with_guard",
    "single_bad_first_step_then_stable",
]

MAX_LENGTH = 128
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5

BASE_LR = 1e-4
LEGACY_INITIAL_LR = 1.0
WARMUP_STEPS = 8000
TOTAL_STEPS = 50000

GUARD_WINDOW_STEPS = 100
DIAGNOSTIC_STEPS = 300

THRESHOLD_MULTIPLIER = 10.0
THRESHOLD_SOURCE = "calibrated_10x_safe_max"

RESULT_ROOT = PROJECT_ROOT / f"results/warmup_safety_gate_{DATASET_NAME}_{RUN_TAG}"
PER_SEED_DIR = RESULT_ROOT / "per_seed"
TRACE_DIR = RESULT_ROOT / "scheduler_traces"
GUARD_DIR = RESULT_ROOT / "guard_events"
EARLY_DIR = RESULT_ROOT / "early_logs"

for d in [RESULT_ROOT, PER_SEED_DIR, TRACE_DIR, GUARD_DIR, EARLY_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def stable_lr(step):
    if step < WARMUP_STEPS:
        scale = step / max(1, WARMUP_STEPS)
    else:
        progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
        progress = min(max(progress, 0.0), 1.0)
        scale = 0.5 * (1.0 + math.cos(math.pi * progress))
    return BASE_LR * scale


def legacy_lr(model, step):
    step = max(1, step)
    d_model = model.d_model
    return (d_model ** -0.5) * min(step ** -0.5, step * (WARMUP_STEPS ** -1.5))


def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group["lr"] = lr


def clone_trainable_params(model):
    return {
        name: p.detach().clone()
        for name, p in model.named_parameters()
        if p.requires_grad
    }


def compute_update_ratios(model, before_params):
    total_update_sq = 0.0
    total_param_sq = 0.0
    layer_update_sq = {}
    layer_param_sq = {}

    for name, p in model.named_parameters():
        if not p.requires_grad or name not in before_params:
            continue

        before = before_params[name].to(p.device)
        after = p.detach()

        upd_sq = torch.sum((after - before) ** 2).item()
        par_sq = torch.sum(before ** 2).item()

        total_update_sq += upd_sq
        total_param_sq += par_sq

        if "embedding" in name:
            group = "embedding"
        elif "classifier" in name:
            group = "classifier"
        elif "encoder_layers" in name:
            parts = name.split(".")
            group = f"layer_{parts[1]}" if len(parts) > 1 else "encoder"
        else:
            group = "other"

        layer_update_sq[group] = layer_update_sq.get(group, 0.0) + upd_sq
        layer_param_sq[group] = layer_param_sq.get(group, 0.0) + par_sq

    global_ratio = np.sqrt(total_update_sq) / (np.sqrt(total_param_sq) + 1e-12)

    layer_ratios = {
        k: np.sqrt(layer_update_sq[k]) / (np.sqrt(layer_param_sq[k]) + 1e-12)
        for k in layer_update_sq
    }

    return float(global_ratio), layer_ratios


def evaluate_full(model, loader, device, num_classes):
    model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    correct = 0
    total = 0

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            total_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)

    acc = correct / total
    ece = expected_calibration_error(labels, probs)
    nll = multiclass_nll(labels, probs)
    brier = multiclass_brier(labels, probs, num_classes)

    return {
        "loss": float(total_loss / len(loader)),
        "acc": float(acc),
        "ece": float(ece),
        "nll": float(nll),
        "brier": float(brier),
    }


def compute_lr_for_condition(condition, model, current_step, lr_before):
    next_step = current_step + 1

    if condition in ["buggy_no_guard", "buggy_failfast_guard", "buggy_rescue_guard"]:
        lr_used = lr_before
        lr_after = legacy_lr(model, next_step)
        return lr_used, lr_after, "schedule_after_step"

    if condition == "fixed_order_with_guard":
        lr_used = legacy_lr(model, next_step)
        lr_after = lr_used
        return lr_used, lr_after, "schedule_before_step"

    if condition == "stable_with_guard":
        lr_used = stable_lr(next_step)
        lr_after = lr_used
        return lr_used, lr_after, "schedule_before_step"

    if condition == "single_bad_first_step_then_stable":
        if current_step == 0:
            lr_used = LEGACY_INITIAL_LR
        else:
            lr_used = stable_lr(next_step)
        lr_after = stable_lr(next_step)
        return lr_used, lr_after, "single_bad_then_stable"

    raise ValueError(f"Unknown condition: {condition}")


def run_one(model_name, condition, seed, threshold, device, calibration_phase=False):
    print("\n" + "=" * 100)
    print(f"RUN: model={model_name} | condition={condition} | seed={seed}")
    print("=" * 100)

    set_seed(seed)

    train_dataset, val_dataset, test_dataset, vocab, num_classes = load_text_dataset(
        DATASET_NAME,
        max_length=MAX_LENGTH,
        split_seed=seed,
        train_fraction=TRAIN_FRACTION,
    )

    train_loader, val_loader, test_loader = get_dataloaders(
        train_dataset,
        val_dataset,
        test_dataset,
        batch_size=BATCH_SIZE,
    )

    cfg = [c for c in create_model_configs() if c["name"] == model_name][0]

    model = TransformerClassifier(
        vocab_size=len(vocab),
        num_classes=num_classes,
        d_model=cfg["d_model"],
        n_layers=cfg["n_layers"],
        n_heads=cfg["n_heads"],
        d_ff=cfg["d_ff"],
        max_seq_len=MAX_LENGTH,
    ).to(device)

    num_parameters = model.count_parameters()

    init_lr = BASE_LR if condition == "stable_with_guard" else LEGACY_INITIAL_LR

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=init_lr,
        betas=(0.9, 0.98),
        eps=1e-9,
    )

    criterion = nn.CrossEntropyLoss()

    scheduler_trace = []
    guard_events = []
    early_logs = []

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    stopped_by_guard = False
    rescued_once = False
    first_guard_trigger_step = None
    current_step = 0
    max_early_update_ratio = 0.0

    for epoch in range(NUM_EPOCHS):
        if stopped_by_guard:
            break

        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        batches_seen = 0

        for batch in train_loader:
            batches_seen += 1

            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids)
            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            before_params = clone_trainable_params(model)
            model_state_before = copy.deepcopy(model.state_dict())
            optim_state_before = copy.deepcopy(optimizer.state_dict())

            lr_before = float(optimizer.param_groups[0]["lr"])
            lr_used, lr_after, schedule_order = compute_lr_for_condition(
                condition, model, current_step, lr_before
            )

            if schedule_order in ["schedule_before_step", "single_bad_then_stable"]:
                set_optimizer_lr(optimizer, lr_used)

            optimizer.step()

            unsafe_update_ratio, unsafe_layer_ratios = compute_update_ratios(model, before_params)

            current_step += 1

            if schedule_order == "schedule_after_step":
                set_optimizer_lr(optimizer, lr_after)

            final_update_ratio = unsafe_update_ratio
            final_layer_ratios = unsafe_layer_ratios
            guard_action = "none"

            if (
                not calibration_phase
                and current_step <= GUARD_WINDOW_STEPS
                and condition in ["buggy_failfast_guard", "buggy_rescue_guard"]
            ):
                if unsafe_update_ratio > threshold:
                    first_guard_trigger_step = first_guard_trigger_step or current_step

                    if condition == "buggy_failfast_guard":
                        guard_action = "failfast_stop"
                        stopped_by_guard = True

                    elif condition == "buggy_rescue_guard":
                        guard_action = "rescue_correct_step"
                        rescued_once = True

                        model.load_state_dict(model_state_before)
                        optimizer.load_state_dict(optim_state_before)

                        safe_lr = legacy_lr(model, current_step)
                        set_optimizer_lr(optimizer, safe_lr)

                        optimizer.step()

                        final_update_ratio, final_layer_ratios = compute_update_ratios(model, before_params)
                        lr_used = safe_lr
                        lr_after = safe_lr

                    guard_events.append({
                        "dataset_name": DATASET_NAME,
                        "model_name": model_name,
                        "condition": condition,
                        "seed": int(seed),
                        "step": int(current_step),
                        "action": guard_action,
                        "threshold": float(threshold),
                        "threshold_source": THRESHOLD_SOURCE,
                        "observed_update_ratio": float(unsafe_update_ratio),
                        "final_update_ratio": float(final_update_ratio),
                    })

            max_early_update_ratio = max(max_early_update_ratio, final_update_ratio)

            scheduler_trace.append({
                "step": int(current_step),
                "lr_before_step": float(lr_before),
                "lr_used": float(lr_used),
                "lr_after_step": float(lr_after),
                "condition": condition,
            })

            if current_step <= DIAGNOSTIC_STEPS:
                probs = torch.softmax(logits.detach(), dim=1)
                early_logs.append({
                    "step": int(current_step),
                    "loss": float(loss.item()),
                    "lr_before_step": float(lr_before),
                    "lr_used": float(lr_used),
                    "lr_after_step": float(lr_after),
                    "update_to_param_ratio": float(final_update_ratio),
                    "unsafe_update_to_param_ratio": float(unsafe_update_ratio),
                    "mean_confidence": float(probs.max(dim=1)[0].mean().item()),
                    "layer_update_to_param_ratio": final_layer_ratios,
                })

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            if stopped_by_guard:
                break

        train_loss = total_loss / max(1, batches_seen)
        train_acc = correct / max(1, total)

        val_eval = evaluate_full(model, val_loader, device, num_classes)
        val_loss = val_eval["loss"]
        val_acc = val_eval["acc"]

        print(
            f"Epoch {epoch + 1:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"stopped_by_guard={stopped_by_guard}"
        )

        if val_loss < best_val_loss and not stopped_by_guard:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            break

    if best_state is not None and not stopped_by_guard:
        model.load_state_dict(best_state)

    train_eval = evaluate_full(model, train_loader, device, num_classes)
    test_eval = evaluate_full(model, test_loader, device, num_classes)

    generalization_gap = train_eval["acc"] - test_eval["acc"]

    file_stub = f"{DATASET_NAME}_{model_name}_{condition}_seed{seed}"

    scheduler_trace_file = TRACE_DIR / f"{file_stub}_scheduler_trace.json"
    guard_events_file = GUARD_DIR / f"{file_stub}_guard_events.json"
    early_logs_file = EARLY_DIR / f"{file_stub}_early_logs.json"

    save_json(scheduler_trace, scheduler_trace_file)
    save_json(guard_events, guard_events_file)
    save_json(early_logs, early_logs_file)

    result = {
        "dataset_name": DATASET_NAME,
        "run_tag": RUN_TAG,
        "run_mode": RUN_MODE,
        "model_name": model_name,
        "condition": condition,
        "seed": int(seed),
        "num_parameters": int(num_parameters),
        "train_fraction": float(TRAIN_FRACTION),
        "num_epochs_config": int(NUM_EPOCHS),
        "threshold": None if threshold is None else float(threshold),
        "threshold_source": THRESHOLD_SOURCE,
        "threshold_multiplier": float(THRESHOLD_MULTIPLIER),
        "threshold_calibration_conditions": CALIBRATION_CONDITIONS,
        "calibration_phase": bool(calibration_phase),
        "test_acc": float(test_eval["acc"]),
        "test_ece": float(test_eval["ece"]),
        "test_nll": float(test_eval["nll"]),
        "test_brier": float(test_eval["brier"]),
        "train_acc": float(train_eval["acc"]),
        "generalization_gap_01": float(generalization_gap),
        "guard_trigger_count": int(len(guard_events)),
        "stopped_by_guard": bool(stopped_by_guard),
        "rescued_once": bool(rescued_once),
        "first_guard_trigger_step": first_guard_trigger_step,
        "max_early_update_ratio": float(max_early_update_ratio),
        "scheduler_trace_file": str(scheduler_trace_file.relative_to(RESULT_ROOT)),
        "guard_events_file": str(guard_events_file.relative_to(RESULT_ROOT)),
        "early_logs_file": str(early_logs_file.relative_to(RESULT_ROOT)),
    }

    per_seed_file = PER_SEED_DIR / f"{file_stub}.json"
    save_json(result, per_seed_file)

    print(
        f"FINAL | test_acc={result['test_acc']:.4f} | "
        f"ece={result['test_ece']:.4f} | "
        f"nll={result['test_nll']:.4f} | "
        f"max_update={result['max_early_update_ratio']:.3e} | "
        f"guard_triggers={result['guard_trigger_count']} | "
        f"stopped={result['stopped_by_guard']} | "
        f"rescued={result['rescued_once']}"
    )

    return result


def calibrate_threshold(device):
    print("\n" + "=" * 100)
    print("CALIBRATING THRESHOLD FROM SAFE RUNS ONLY")
    print("=" * 100)

    calibration_results = []

    for model_name in MODEL_NAMES:
        for condition in CALIBRATION_CONDITIONS:
            for seed in SEEDS:
                result = run_one(
                    model_name=model_name,
                    condition=condition,
                    seed=seed,
                    threshold=None,
                    device=device,
                    calibration_phase=True,
                )
                calibration_results.append(result)

                pd.DataFrame(calibration_results).to_csv(
                    RESULT_ROOT / "threshold_calibration_safe_runs.csv",
                    index=False,
                )

    safe_max = max(r["max_early_update_ratio"] for r in calibration_results)
    threshold = THRESHOLD_MULTIPLIER * safe_max

    calibration_record = {
        "threshold": float(threshold),
        "threshold_source": THRESHOLD_SOURCE,
        "threshold_multiplier": float(THRESHOLD_MULTIPLIER),
        "safe_max_update_ratio": float(safe_max),
        "threshold_calibration_conditions": CALIBRATION_CONDITIONS,
        "calibration_models": MODEL_NAMES,
        "calibration_seeds": SEEDS,
        "calibration_rule": "threshold = 10 * max(max_early_update_ratio over stable_with_guard and fixed_order_with_guard)",
    }

    save_json(calibration_record, RESULT_ROOT / "threshold_calibration_record.json")

    print("\nCALIBRATION COMPLETE")
    print(json.dumps(calibration_record, indent=2))

    return threshold, calibration_record, calibration_results


def summarize_results(results, threshold):
    rows = []

    for (model_name, condition), sub_df in pd.DataFrame(results).groupby(["model_name", "condition"]):
        rows.append({
            "dataset_name": DATASET_NAME,
            "model_name": model_name,
            "condition": condition,
            "num_parameters": int(sub_df["num_parameters"].iloc[0]),
            "seeds": sorted(sub_df["seed"].astype(int).tolist()),
            "threshold": float(threshold),
            "threshold_source": THRESHOLD_SOURCE,
            "threshold_multiplier": float(THRESHOLD_MULTIPLIER),
            "threshold_calibration_conditions": CALIBRATION_CONDITIONS,
            "guard_trigger_rate": float((sub_df["guard_trigger_count"] > 0).mean()),
            "stopped_rate": float(sub_df["stopped_by_guard"].mean()),
            "rescued_rate": float(sub_df["rescued_once"].mean()),
            "first_guard_trigger_step_mean": float(sub_df["first_guard_trigger_step"].dropna().mean()) if sub_df["first_guard_trigger_step"].notna().any() else None,
            "max_early_update_ratio_mean": float(sub_df["max_early_update_ratio"].mean()),
            "test_acc_stats": mean_ci95(sub_df["test_acc"].values),
            "ece_stats": mean_ci95(sub_df["test_ece"].values),
            "nll_stats": mean_ci95(sub_df["test_nll"].values),
            "brier_stats": mean_ci95(sub_df["test_brier"].values),
            "gap_stats": mean_ci95(sub_df["generalization_gap_01"].values),
        })

    return rows


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    threshold, calibration_record, calibration_results = calibrate_threshold(device)

    manifest = {
        "dataset_name": DATASET_NAME,
        "run_tag": RUN_TAG,
        "run_mode": RUN_MODE,
        "models": MODEL_NAMES,
        "conditions": CONDITIONS,
        "calibration_conditions": CALIBRATION_CONDITIONS,
        "post_calibration_conditions": POST_CALIBRATION_CONDITIONS,
        "seeds": SEEDS,
        "train_fraction": TRAIN_FRACTION,
        "base_lr": BASE_LR,
        "legacy_initial_lr": LEGACY_INITIAL_LR,
        "warmup_steps": WARMUP_STEPS,
        "total_steps": TOTAL_STEPS,
        "guard_window_steps": GUARD_WINDOW_STEPS,
        "diagnostic_steps": DIAGNOSTIC_STEPS,
        "threshold": float(threshold),
        "threshold_source": THRESHOLD_SOURCE,
        "threshold_multiplier": float(THRESHOLD_MULTIPLIER),
        "safe_max_update_ratio": float(calibration_record["safe_max_update_ratio"]),
        "threshold_calibration_conditions": CALIBRATION_CONDITIONS,
        "threshold_calibration_rule": calibration_record["calibration_rule"],
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "device": device,
    }

    save_json(manifest, RESULT_ROOT / "run_manifest.json")

    print("\n" + "=" * 100)
    print("DBPEDIA WARMUP SAFETY GATE MANIFEST")
    print("=" * 100)
    for k, v in manifest.items():
        print(f"{k:<35}: {v}")

    results = list(calibration_results)

    pd.DataFrame(results).to_csv(
        RESULT_ROOT / "warmup_safety_gate_per_seed.csv",
        index=False,
    )

    save_json(
        summarize_results(results, threshold),
        RESULT_ROOT / "warmup_safety_gate_summary.json",
    )

    for model_name in MODEL_NAMES:
        for condition in POST_CALIBRATION_CONDITIONS:
            for seed in SEEDS:
                result = run_one(
                    model_name=model_name,
                    condition=condition,
                    seed=seed,
                    threshold=threshold,
                    device=device,
                    calibration_phase=False,
                )

                results.append(result)

                pd.DataFrame(results).to_csv(
                    RESULT_ROOT / "warmup_safety_gate_per_seed.csv",
                    index=False,
                )

                save_json(
                    summarize_results(results, threshold),
                    RESULT_ROOT / "warmup_safety_gate_summary.json",
                )

    final_summary = summarize_results(results, threshold)

    print("\nSUMMARY")
    print(json.dumps(final_summary, indent=2))

    print("\nSaved result root:", RESULT_ROOT)


if __name__ == "__main__":
    main()

### Run DBPedia Warmup Safety Gate script

In [ ]:
!python transformer_project/run_warmup_safety_gate_dbpedia.py

### Analyze DBpedia Warmup Safety Gate results

In [ ]:
import json
import zipfile
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files

# ============================================================
# UPLOAD / EXTRACT ZIP IF NEEDED
# ============================================================

extract_dir = Path("/content/results_extracted")
extract_dir.mkdir(parents=True, exist_ok=True)

# If extracted folder is empty or DBPedia is missing, upload zip again
existing_db = list(extract_dir.glob("**/warmup_safety_gate_summary.json"))

if len(existing_db) == 0:
    print("Upload the consolidated artifact ZIP now...")
    uploaded = files.upload()

    zip_files = [Path("/content") / name for name in uploaded.keys() if name.endswith(".zip")]
    if not zip_files:
        raise FileNotFoundError("No zip file uploaded.")

    zip_path = zip_files[0]
    print("Using zip:", zip_path)

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    print("Extracted to:", extract_dir)

# ============================================================
# FIND DBPEDIA ROOT
# ============================================================

candidate_roots = []

for p in extract_dir.rglob("run_manifest.json"):
    root = p.parent
    try:
        with open(p, "r") as f:
            manifest_test = json.load(f)

        dataset_name = str(manifest_test.get("dataset_name", "")).lower()
        run_tag = str(manifest_test.get("run_tag", "")).lower()

        if dataset_name == "dbpedia_14" and "warmup_safety_gate" in run_tag:
            candidate_roots.append(root)

    except Exception:
        pass

if not candidate_roots:
    print("Available run_manifest.json files:")
    for p in extract_dir.rglob("run_manifest.json"):
        print(p)
    raise FileNotFoundError("Could not find DBPedia Warmup Safety Gate result root.")

DB_GUARD_ROOT = candidate_roots[0]

print("Using DB_GUARD_ROOT:")
print(DB_GUARD_ROOT)

# ============================================================
# ANALYSIS SETUP
# ============================================================

DB_ANALYSIS_DIR = Path("/content/dbpedia_warmup_safety_gate_analysis")
DB_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def save_table(df, filename):
    path = DB_ANALYSIS_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)

def save_plot(filename):
    path = DB_ANALYSIS_DIR / filename
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved:", path)

# ============================================================
# LOAD RESULTS
# ============================================================

manifest = load_json(DB_GUARD_ROOT / "run_manifest.json")
summary = load_json(DB_GUARD_ROOT / "warmup_safety_gate_summary.json")
per_seed_df = pd.read_csv(DB_GUARD_ROOT / "warmup_safety_gate_per_seed.csv")
calibration_df = pd.read_csv(DB_GUARD_ROOT / "threshold_calibration_safe_runs.csv")
calibration_record = load_json(DB_GUARD_ROOT / "threshold_calibration_record.json")

print("\nDBPEDIA WARMUP SAFETY GATE MANIFEST")
print("=" * 100)
for k, v in manifest.items():
    print(f"{k:<35}: {v}")

print("\nTHRESHOLD CALIBRATION RECORD")
print("=" * 100)
for k, v in calibration_record.items():
    print(f"{k:<35}: {v}")

print("\nDBPEDIA PER-SEED RESULTS")
display(per_seed_df)

print("\nDBPEDIA CALIBRATION SAFE RUNS")
display(calibration_df)

save_table(per_seed_df, "dbpedia_warmup_safety_gate_per_seed.csv")
save_table(calibration_df, "dbpedia_threshold_calibration_safe_runs.csv")

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_rows = []

for row in summary:
    summary_rows.append({
        "model_name": row["model_name"],
        "condition": row["condition"],
        "num_parameters": row["num_parameters"],
        "threshold": row["threshold"],
        "threshold_source": row["threshold_source"],
        "threshold_multiplier": row["threshold_multiplier"],
        "threshold_calibration_conditions": row["threshold_calibration_conditions"],
        "guard_trigger_rate": row["guard_trigger_rate"],
        "stopped_rate": row["stopped_rate"],
        "rescued_rate": row["rescued_rate"],
        "first_guard_trigger_step_mean": row["first_guard_trigger_step_mean"],
        "max_early_update_ratio_mean": row["max_early_update_ratio_mean"],
        "test_acc_mean": row["test_acc_stats"]["mean"],
        "test_acc_std": row["test_acc_stats"]["std"],
        "test_acc_ci95_low": row["test_acc_stats"]["ci95_low"],
        "test_acc_ci95_high": row["test_acc_stats"]["ci95_high"],
        "ece_mean": row["ece_stats"]["mean"],
        "nll_mean": row["nll_stats"]["mean"],
        "brier_mean": row["brier_stats"]["mean"],
        "gap_mean": row["gap_stats"]["mean"],
    })

summary_df = pd.DataFrame(summary_rows)

condition_order = [
    "buggy_no_guard",
    "buggy_failfast_guard",
    "buggy_rescue_guard",
    "stable_with_guard",
    "fixed_order_with_guard",
    "single_bad_first_step_then_stable",
]

summary_df["condition"] = pd.Categorical(
    summary_df["condition"],
    categories=condition_order,
    ordered=True,
)

summary_df = summary_df.sort_values(["model_name", "condition"]).reset_index(drop=True)

print("\nDBPEDIA SUMMARY TABLE")
display(summary_df)

save_table(summary_df, "dbpedia_warmup_safety_gate_summary.csv")

# ============================================================
# PLOTS
# ============================================================

for model_name in sorted(summary_df["model_name"].unique()):
    sub = summary_df[summary_df["model_name"] == model_name].copy()

    plt.figure(figsize=(10, 5))
    plt.bar(sub["condition"].astype(str), sub["test_acc_mean"], yerr=sub["test_acc_std"], capsize=4)
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Test accuracy")
    plt.title(f"DBPedia Warmup Safety Gate — Accuracy — {model_name}")
    plt.grid(axis="y", alpha=0.3)
    save_plot(f"dbpedia_guard_accuracy_{model_name}.png")
    plt.show()

for model_name in sorted(summary_df["model_name"].unique()):
    sub = summary_df[summary_df["model_name"] == model_name].copy()

    plt.figure(figsize=(10, 5))
    plt.bar(sub["condition"].astype(str), sub["max_early_update_ratio_mean"])
    plt.axhline(float(manifest["threshold"]), linestyle="--", linewidth=2, label="guard threshold")
    plt.yscale("log")
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Mean max early update/parameter ratio")
    plt.title(f"DBPedia Warmup Safety Gate — Early Update Ratio — {model_name}")
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    save_plot(f"dbpedia_guard_update_ratio_{model_name}.png")
    plt.show()

for model_name in sorted(summary_df["model_name"].unique()):
    sub = summary_df[summary_df["model_name"] == model_name].copy()

    plt.figure(figsize=(10, 5))
    plt.bar(sub["condition"].astype(str), sub["nll_mean"])
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("NLL")
    plt.title(f"DBPedia Warmup Safety Gate — NLL — {model_name}")
    plt.grid(axis="y", alpha=0.3)
    save_plot(f"dbpedia_guard_nll_{model_name}.png")
    plt.show()

# ============================================================
# ZIP ANALYSIS ARTIFACTS
# ============================================================

zip_path = Path("/content/dbpedia_warmup_safety_gate_analysis.zip")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in DB_ANALYSIS_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(DB_ANALYSIS_DIR)))

print("Created:", zip_path)
files.download(str(zip_path))